# S3 J1 — Architectures multi-agents

Notebook étudiant généré depuis le Markdown source.

# Objectifs d’apprentissage

## Objectifs principaux

À la fin de cette journée, l’apprenant sera capable de :

1. expliquer les limites d’un agent mono-agent autonome ;
2. identifier les cas où une architecture multi-agents est justifiée ;
3. comparer manager-worker, handoff, router, parallèle et reviewer ;
4. concevoir un contrat minimal entre agents ;
5. définir des responsabilités sans chevauchement ;
6. implémenter une simulation multi-agents déterministe ;
7. produire des traces d’exécution exploitables ;
8. évaluer les risques de coût, latence, boucle et dilution de responsabilité.

## Compétences AI Engineering

- Décomposition de problème.
- Design d’orchestration.
- Séparation des responsabilités.
- Contrats d’entrée/sortie.
- Validation d’architecture.
- Observabilité.
- Tests sans dépendance LLM.

## Limites de la journée

Cette journée ne couvre pas encore :

- serveur MCP ;
- client MCP ;
- synchronisation avancée d’état ;
- évaluation quantitative complète.

## Synthèse

Un système multi-agents est une architecture logicielle composée d’agents spécialisés, de contrats, d’un routage, d’un état partagé contrôlé et de traces.

## Patterns

- Manager-worker
- Handoff
- Router
- Parallèle
- Reviewer

In [ ]:
from dataclasses import dataclass
from typing import Tuple

@dataclass(frozen=True)
class AgentSpec:
    name: str
    role: str
    keywords: Tuple[str, ...]

    def matches(self, message: str) -> int:
        return sum(1 for keyword in self.keywords if keyword.lower() in message.lower())

agents = [
    AgentSpec("billing", "Billing Specialist", ("invoice", "payment", "subscription")),
    AgentSpec("technical", "Technical Specialist", ("bug", "error", "crash")),
]

message = "I have a payment issue with my invoice"
scores = {agent.name: agent.matches(message) for agent in agents}
scores

## Lab

Depuis `book/week03/day01/labs/` :

```bash
python multi_agent_architecture.py
python test_multi_agent_architecture.py
```

# Exercices — Architectures multi-agents

## Exercice 1 — Identifier le bon pattern

Choisir le pattern adapté :

1. Router une demande client vers facturation, technique ou remboursement.
2. Produire un rapport final à partir de trois analyses spécialisées.
3. Vérifier qu’une réponse respecte une politique interne.
4. Transférer la conversation à un spécialiste juridique.
5. Analyser un document sous trois angles indépendants.

Justifier chaque réponse.

## Exercice 2 — Définir des responsabilités

Concevoir une architecture multi-agents pour une plateforme SaaS B2B avec :

- facturation ;
- bug applicatif ;
- demande de fonctionnalité ;
- sécurité ;
- résiliation.

Inclure un router, au moins trois spécialistes et un reviewer.

## Exercice 3 — Contrat d’agent

Écrire un contrat JSON pour `security_specialist` avec :

- nom ;
- responsabilité ;
- entrées ;
- sortie ;
- outils ;
- conditions de handoff ;
- modes d’échec.

## Exercice 4 — État partagé

Classer ces éléments :

- toujours partagé ;
- partagé sous condition ;
- jamais partagé.

Éléments :

1. message utilisateur courant ;
2. historique complet ;
3. décision de routage ;
4. mémoire long terme ;
5. résultat intermédiaire ;
6. données sensibles ;
7. trace d’exécution ;
8. raisonnement interne.

## Exercice 5 — Limites du multi-agent

Donner trois cas où il vaut mieux garder un agent unique. Expliquer le risque créé par le multi-agent.

## Exercice 6 — Lab

Exécuter :

```bash
python multi_agent_architecture.py
python test_multi_agent_architecture.py
```

Puis ajouter un spécialiste `security` et un test de routage.

# Challenge — Assistant multi-agents de support SaaS

## Contexte

Une entreprise SaaS veut traiter des tickets entrants :

- facturation ;
- bug technique ;
- sécurité ;
- demande produit ;
- remboursement ;
- demande ambiguë.

L’assistant doit router, déléguer, produire une réponse, demander clarification si nécessaire, faire valider la réponse et enregistrer une trace.

## Mission

Concevoir une architecture multi-agents complète.

## Livrables attendus

1. Schéma d’architecture.
2. Liste des agents.
3. Rôle de chaque agent.
4. Entrées/sorties de chaque agent.
5. Règles de routage.
6. Règles de handoff.
7. Règles de partage d’état.
8. Erreurs possibles.
9. Métriques de qualité.
10. Stratégie de test.

## Contraintes

- Maximum six agents.
- Rôles distincts.
- Reviewer limité à la validation.
- Données sensibles non partagées par défaut.
- Demandes ambiguës → clarification.
- Décisions de routage structurées.

## Critères de réussite

Le design est réussi si les responsabilités sont non ambiguës, les interactions traçables, les limites explicites et les tests réalisables sans LLM réel.

# Corrections — Notebook formateur

# Corrigé — Exercices

## Exercice 1

1. **Router / triage** : il faut choisir le bon domaine.
2. **Manager-worker** : le manager agrège plusieurs analyses.
3. **Reviewer** : il vérifie la conformité d’une réponse.
4. **Handoff** : le spécialiste doit prendre le contrôle.
5. **Parallèle** : les analyses sont indépendantes.

## Exercice 2

Architecture possible :

```text
User → Triage Agent
        ├── Billing Specialist
        ├── Technical Specialist
        ├── Product Specialist
        ├── Security Specialist
        └── Cancellation Specialist
              ↓
           Reviewer Agent
              ↓
          Final Answer
```

Responsabilités :

- `triage_agent` : classer la demande et choisir l’agent cible.
- `billing_specialist` : factures, paiements, abonnements.
- `technical_specialist` : bugs, incidents, logs.
- `product_specialist` : demandes de fonctionnalités.
- `security_specialist` : accès, permissions, conformité.
- `cancellation_specialist` : résiliation.
- `reviewer_agent` : vérifier format, ton, limites et sécurité.

## Exercice 3

```json
{
  "agent_name": "security_specialist",
  "responsibility": "Analyze security-related customer requests and produce a safe support answer.",
  "input_schema": {
    "ticket_id": "string",
    "customer_message": "string",
    "account_context": "object"
  },
  "output_schema": {
    "category": "security",
    "answer": "string",
    "risk_level": "low|medium|high",
    "requires_human_review": "boolean"
  },
  "allowed_tools": [
    "get_security_policy",
    "get_account_security_events"
  ],
  "handoff_conditions": [
    "legal_request",
    "suspected_data_breach",
    "law_enforcement_request"
  ],
  "failure_modes": [
    "missing_account_context",
    "insufficient_authorization",
    "ambiguous_security_request"
  ]
}
```

## Exercice 4

| Élément | Classification |
|---|---|
| Message utilisateur courant | Toujours partagé |
| Historique complet | Partagé sous condition |
| Décision de routage | Toujours partagé |
| Mémoire long terme | Partagé sous condition stricte |
| Résultat intermédiaire | Partagé sous condition |
| Données sensibles | Partagé sous condition stricte |
| Trace d’exécution | Toujours partagé côté système |
| Raisonnement interne | Jamais partagé |

## Exercice 5

- FAQ simple : coût et latence inutiles.
- Tâche mono-domaine : routage sans bénéfice.
- Rôles flous : réponses contradictoires et responsabilités diluées.

## Exercice 6

Une bonne extension ajoute un agent `security`, ses mots-clés et un test déterministe. Le test ne doit pas dépendre d’un appel LLM réel.

# Corrigé — Questions d’entretien

## 1

Parce qu’une architecture multi-agents ajoute coût, latence, complexité, risques de boucle et difficulté de debug. Elle doit être justifiée par une séparation réelle des responsabilités.

## 2

Manager-worker : le manager garde le contrôle et agrège les résultats.  
Handoff : le contrôle est transféré à un spécialiste qui devient responsable de la suite.

## 3

Un reviewer est utile pour vérifier conformité, sécurité, format, complétude et ton. Il valide ou rejette, mais ne remplace pas le spécialiste métier.

## 4

Un état partagé trop large peut exposer des données sensibles, ajouter du bruit contextuel, créer des influences non souhaitées et compliquer l’audit.

## 5

Avec une limite d’itérations, des statuts terminaux, des règles de handoff explicites, un historique des agents appelés et un fallback vers humain ou clarification.

## 6

Tracer : session, étape, agent, action, entrée résumée, sortie structurée, décision, statut et erreur éventuelle.

## 7

Appel comme outil si le manager garde la réponse finale. Handoff si le spécialiste doit prendre le contrôle de la conversation.

## 8

Les sorties structurées rendent l’intégration testable, validable et observable.

## 9

Un router mal conçu produit mauvais handoffs, clarifications manquées, coûts inutiles et mauvaise expérience utilisateur.

## 10

Avec des agents mockés, règles déterministes, tests unitaires, contrats JSON, scénarios d’erreur et snapshots de traces.

# Corrigé — Challenge

## Schéma

```mermaid
flowchart TD
    U[Utilisateur] --> T[Triage Agent]
    T -->|billing| B[Billing Specialist]
    T -->|technical| E[Technical Specialist]
    T -->|security| S[Security Specialist]
    T -->|product| P[Product Specialist]
    T -->|refund| R[Refund Specialist]
    T -->|ambiguous| C[Clarification]
    B --> V[Reviewer Agent]
    E --> V
    S --> V
    P --> V
    R --> V
    V --> F[Réponse finale]
```

## Agents

1. `triage_agent`
2. `billing_specialist`
3. `technical_specialist`
4. `security_specialist`
5. `product_refund_specialist`
6. `reviewer_agent`

## Règles de routage

- `invoice`, `payment`, `subscription` → billing.
- `bug`, `error`, `crash`, `latency` → technical.
- `access`, `permission`, `breach`, `security` → security.
- `feature`, `roadmap`, `improvement`, `refund`, `cancel` → product/refund.
- Score faible ou plusieurs catégories → clarification.

## Partage d’état

Toujours partagé :

- message courant ;
- décision de routage ;
- trace.

Sous condition :

- historique ;
- contexte compte ;
- résultat intermédiaire.

Jamais partagé :

- secrets système ;
- raisonnement interne ;
- données sensibles sans besoin strict.

## Métriques

- taux de routage correct ;
- taux de clarification ;
- taux d’escalade humaine ;
- temps de résolution ;
- coût par ticket ;
- taux de rejet reviewer ;
- satisfaction utilisateur.

## Stratégie de test

- tests unitaires du router ;
- tests de contrats JSON ;
- tests de demandes ambiguës ;
- tests de reviewer ;
- tests anti-boucles ;
- tests de non-partage de données sensibles.

## Propositions d’amélioration

Ajouter un dataset de tickets annotés pour mesurer offline la qualité du routage avant mise en production.

# Review formateur — Jour 1 Semaine 3

## Intention pédagogique

Cette journée marque le passage de l’agent autonome vers une architecture composée. Le message central : le multi-agent est un compromis d’architecture, pas une amélioration automatique.

## Messages clés

1. Un système multi-agents doit être justifié.
2. Chaque agent doit avoir un rôle clair.
3. Les contrats sont plus importants que les prompts.
4. Le routage est critique.
5. L’état partagé doit rester minimal.
6. Les traces sont indispensables.
7. On peut tester sans LLM réel.

## Points d’attention

Surveiller les conceptions où :

- tous les agents peuvent tout faire ;
- le reviewer réécrit tout ;
- les données sensibles sont partagées partout ;
- aucun statut terminal n’existe ;
- aucune règle anti-boucle n’est prévue ;
- le router ne produit pas de sortie structurée.

## Débrief du lab

Le lab est déterministe volontairement. L’objectif est de rendre testables les décisions d’architecture : routage, handoff, reviewer, traces et extension.

## Transition vers le jour 2

Le jour 2 traitera la coordination : enchaîner les agents, gérer contradictions, séquencement, synchronisation et stratégies de consolidation.